In [4]:
from enum import Enum
import random
from collections import namedtuple, deque


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.multiprocessing as mp

from engine import *
from vnn import *

## defaults

In [5]:
EPOCHS = 100
BATCH_SIZE = 30_000
NEW_MOVES_TO_ADD_PER_EPOCH = 1_000
EPS_START = 0.9
EPS_END = 0.0001
EPS_DECAY = 10
LR = 3e-4

# Training

In [6]:
class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, game_state: Game_state):
        """Save a transition"""
        self.memory.append(game_state)

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

In [7]:
vnn = VNN()
vnn.share_memory()
optimizer = optim.AdamW(vnn.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10 * BATCH_SIZE)

trajectories_by_epoch = dict()

In [10]:
with mp.Pool(10) as pool:
    trajectories = pool.map(vnn.play_game_and_get_trajectory, [0.9] * 100)

In [14]:
sum([len(t) for t in trajectories])

1535

In [ ]:
vnn.play_game_and_get_trajectory(eps=0.9)

In [ ]:
trajectories

In [ ]:
for epoch in range(EPOCHS):
    print("Epoch:", epoch)

    trajectories_from_this_epoch = []

    eps_threshold = EPS_END + (EPS_START - EPS_END) * np.exp(-1.0 * epoch / EPS_DECAY)
    print("eps", eps_threshold)
    while (
        sum([len(traj) for traj in trajectories_from_this_epoch])
        < NEW_MOVES_TO_ADD_PER_EPOCH
    ):
        trajectories_from_this_epoch.append(
            vnn.play_game_and_get_trajectory(eps=eps_threshold)
        )

    for traj in trajectories_from_this_epoch:
        for single_round in traj:
            memory.push(single_round)

    trajectories_by_epoch[epoch] = trajectories_from_this_epoch

    print(
        "Average lengths of games",
        np.array([len(traj) for traj in trajectories_from_this_epoch]).mean(),
    )

    state_sample_from_memory = memory.sample(min(BATCH_SIZE, len(memory)))
    boards = [state.board for state in state_sample_from_memory]

    target = [state.implied_value_from_best_move for state in state_sample_from_memory]

    criterion = nn.SmoothL1Loss()
    loss = criterion(
        vnn.forward_from_int_boards(boards).reshape(-1), torch.tensor(target)
    )

    optimizer.zero_grad()
    loss.backward()

    torch.nn.utils.clip_grad_value_(vnn.parameters(), 100)
    optimizer.step()

In [ ]:
def show_board(board):
    log_board = np.zeros_like(board, dtype="float32")

    log_board[board != 0] = (np.log2(board[board != 0]) + 10) / 20

    plt.imshow(log_board, cmap="viridis", vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            val = board[i, j]
            if val != 0:
                plt.text(j, i, str(val), ha="center", va="center", color="white")
    plt.xticks([])
    plt.yticks([])
    plt.show()


def visualize_sequence(boards):
    @interact(t=(0, len(boards) - 1))
    def _view(t=0):
        show_board(boards[t])

In [ ]:
visualize_sequence([state.board for state in trajectories_by_epoch[99][1]])